In [0]:
eventhub_namespace = "eh-vb-ecommerce-dev"
eventhub_name = "ecommerce_orders"
consumer_group = "$Default"

bootstrap_server = (
    f"{eventhub_namespace}.servicebus.windows.net:9093"
)

bronze_path = (
    "abfss://landing@adlsvbecommercedev.dfs.core.windows.net/"
    "ecommerce/kafka/bronze/ecommerce_orders"
)

checkpoint_path = (
    "abfss://landing@adlsvbecommercedev.dfs.core.windows.net/"
    "ecommerce/kafka/checkpoint/ecommerce_orders"
)


In [0]:
EH_CONN_STR = dbutils.secrets.get(
    scope="vb-ecommerce-secrets",
    key="eventhub-connection-string"
)

In [0]:
kafka_options = {
    "kafka.bootstrap.servers": bootstrap_server,
    "subscribe": eventhub_name,
    "startingOffsets": "earliest",

    "kafka.security.protocol": "SASL_SSL",
    "kafka.sasl.mechanism": "PLAIN",

    "kafka.sasl.jaas.config":
        f'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required '
        f'username="$ConnectionString" password="{EH_CONN_STR}";'
}

In [0]:
df_kafka = (
    spark.readStream
         .format("kafka")
         .options(**kafka_options)
         .load()
)

In [0]:
from pyspark.sql.types import (
    StructType, StructField,
    IntegerType, DoubleType, StringType
)
from pyspark.sql.functions import (
    col, from_json,
    current_timestamp, lit
)

order_schema = StructType([
    StructField("order_id", IntegerType(), True),
    StructField("customer_id", IntegerType(), True),
    StructField("product_id", IntegerType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("order_amount", DoubleType(), True),
    StructField("order_status", StringType(), True)
])

df_orders = (
    df_kafka
    .select(
        col("topic"),
        col("partition"),
        col("offset"),
        col("timestamp"),
        col("value").cast("string").alias("json_value")
    )
    .withColumn("data", from_json(col("json_value"), order_schema))
    .select(
        "topic",
        "partition",
        "offset",
        "timestamp",
        "data.*"
    )
)

df_bronze = (
    df_orders
    .withColumn("_ingestion_timestamp", current_timestamp())
    .withColumn("_source", lit("EVENT_HUB_KAFKA"))
)

In [0]:
query = (
    df_bronze.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .start(bronze_path)
)

print("Kafka → Bronze streaming started")